# Setup

In [1]:
# Setup
import json
import os
import sys
from datetime import datetime, timedelta
import time
from dotenv import load_dotenv, set_key
import random
from pathlib import Path

# hashing for signing
import hashlib
import hmac

# requests
import requests

# data munging
import pandas as pd
import numpy as np

# helper functions
from tiktok_api_helpers import *

## API Setup

In [2]:
new_access_token = False
use_refresh_token = False

In [3]:
if (new_access_token | use_refresh_token):

    # get new acces_token
    token_url = "https://auth.tiktok-shops.com/api/v2/token/get"

    if use_refresh_token:
        auth_code = os.environ.get("TIKTOK_REFRESH_TOKEN")
        
    params = {
        "app_key": app_key,
        "app_secret": app_secret,
        "auth_code": auth_code,
        "grant_type": "authorized_code",
    }
    
    response = requests.get(token_url, params=params, timeout=15)
    data = response.json()['data']
    access_token = data['access_token']
    print(data)

    # save tokens in .env file
    set_key(".env", "TIKTOK_ACCESS_TOKEN", data['access_token'])
    set_key(".env", "TIKTOK_REFRESH_TOKEN", data['refresh_token'])

else:
    access_token = os.environ.get("TIKTOK_ACCESS_TOKEN")
    refresh_token = os.environ.get("TIKTOK_REFRESH_TOKEN")

# Step 1. Monitor Sample Application Status

PENDING, AWAITING_SHIPMENT, SHIPPED, CONTENT_PENDING, REJECT_CANCELLED, OVERDUE_CANCELLED, UNFULFILL_CANCELLED, DEL_OPEN_COLLAB, SELLER_NOT_SHIP_CANCELLED, WITHDRAW_CANCELLED, UNFULFILLABLE_CANCELLED, OPS_CANCELLED, OPS_FAILED, OPS_COMPLETED, COMPLETED

In [4]:
statuses = [
    "AWAITING_SHIPMENT", "SHIPPED", "CONTENT_PENDING", 
    "OPS_COMPLETED", "COMPLETED"
]

all_sample_applications = []
for status in statuses:
    print(f"Fetching status: {status}")
    results = search_all_sample_applications(status=status)
    all_sample_applications.extend(results)

print(f"\nTotal combined results: {len(all_sample_applications)}")

Fetching status: AWAITING_SHIPMENT
Page 1: 11 application(s) (total so far: 11)
Fetching status: SHIPPED
Page 1: 50 application(s) (total so far: 50)
Page 2: 17 application(s) (total so far: 67)
Fetching status: CONTENT_PENDING
Page 1: 50 application(s) (total so far: 50)
Page 2: 23 application(s) (total so far: 73)
Fetching status: OPS_COMPLETED
Page 1: 0 application(s) (total so far: 0)
Fetching status: COMPLETED
Page 1: 44 application(s) (total so far: 44)

Total combined results: 195


In [5]:
df_all_sample_applications = pd.json_normalize(all_sample_applications)

In [6]:
df_all_sample_applications.to_csv(SAMPLE_APPLICATIONS_CSV, index=False)

In [7]:
df_all_sample_applications['status'].value_counts()

status
CONTENT_PENDING      73
SHIPPED              67
COMPLETED            44
AWAITING_SHIPMENT    11
Name: count, dtype: int64

# Step 2. Create Conversations

## Check existing conversations

In [8]:
ALL_CONVERSATIONS_CSV = "messaging/all_conversations.csv"
CONVERSATIONS_MANIFEST_CSV = "messaging/creators_conversations_manifest.csv"

In [9]:
all_conversations = []
page_token = ""
page_num = 1

while True:
    result = get_conversation_list(
        page_size=50,
        page_token=page_token,
        conversation_status="ALL",
        only_need_conversation_id=False,
    )

    if result.get("code") != 0:
        print(f"  ⚠️  Page {page_num} failed, stopping here: {result}")
        break

    data = result.get("data", {}) or {}
    conversations = data.get("conversations", [])
    all_conversations.extend(conversations)

    print(f"Page {page_num}: {len(conversations)} conversation(s) (total so far: {len(all_conversations)})")

    if not data.get("has_more"):
        break

    page_token = data.get("next_page_token", "")
    if not page_token:
        break

    page_num += 1
    time.sleep(0.5)

print(f"\nDone. {len(all_conversations)} total conversation(s) collected.")

Page 1: 50 conversation(s) (total so far: 50)
Page 2: 50 conversation(s) (total so far: 100)
Page 3: 50 conversation(s) (total so far: 150)
Page 4: 50 conversation(s) (total so far: 200)
Page 5: 50 conversation(s) (total so far: 250)
Page 6: 50 conversation(s) (total so far: 300)
Page 7: 50 conversation(s) (total so far: 350)
Page 8: 50 conversation(s) (total so far: 400)
Page 9: 50 conversation(s) (total so far: 450)
Page 10: 50 conversation(s) (total so far: 500)
Page 11: 50 conversation(s) (total so far: 550)
Page 12: 50 conversation(s) (total so far: 600)
Page 13: 50 conversation(s) (total so far: 650)
Page 14: 50 conversation(s) (total so far: 700)
Page 15: 50 conversation(s) (total so far: 750)
Page 16: 50 conversation(s) (total so far: 800)
Page 17: 50 conversation(s) (total so far: 850)
Page 18: 50 conversation(s) (total so far: 900)
Page 19: 50 conversation(s) (total so far: 950)
Page 20: 50 conversation(s) (total so far: 1000)
Page 21: 50 conversation(s) (total so far: 1050)


In [10]:
df_all_conversations = pd.DataFrame(all_conversations)

In [11]:
df_all_conversations.to_csv(ALL_CONVERSATIONS_CSV, index=False)

## Track conversation IDs of all creators

In [12]:
# load all creators
df_creators_list = pd.read_excel("all_creators_sorted.xlsx", sheet_name="LIST_CREATOR", usecols=[1, 2, 25])
df_creators = pd.read_csv(CONSOLIDATED_CSV)
df_target_collab = pd.read_csv(TARGET_COLLAB_CREATORS_CSV)

In [13]:
# merge all extracted data
df_creators_messaging = df_creators_list \
    .merge(df_creators[['username', 'creator_open_id']], how='left', on="username") \
    .merge(df_target_collab.rename(columns={'name':'target_collaboration_name'}).drop(['batch_name'], axis=1), how='left', on="creator_open_id") \
    .merge(df_all_sample_applications[['creator.creator_open_id', 'status']].rename(columns={'creator.creator_open_id':'creator_open_id', 'status':'sample_status'}), how='left', on="creator_open_id") \
    .merge(df_all_conversations[['creator_im_id', 'id', 'username']].rename(columns={'id':'conversation_id'}), how='left', on="username")

## [Optional] Create Conversation if invited to collab

In [50]:
list_create_conversation = df_creators_messaging.loc[df_creators_messaging['target_collaboration_id'].notnull() & df_creators_messaging['conversation_id'].isnull(), 'creator_open_id'].tolist()
# list_create_conversation = df_creators_messaging.loc[df_creators_messaging['sample_status'].notnull() & df_creators_messaging['conversation_id'].isnull(), 'creator_open_id'].tolist()
print(f"Generating conversation IDs for {len(list_create_conversation)} new creators.")

Generating conversation IDs for 5 new creators.


In [51]:
conversation_rows = []
failed_conversations = []

In [52]:
for i, creator_open_id in enumerate(list_create_conversation, start=1):
    result = create_conversation_with_creator(creator_open_id, only_need_conversation_id=False)

    if result.get("code") != 0:
        print(f"  ⚠️  Failed: {result}")
        failed_conversations.append(creator_open_id)
    else:
        data = result["data"]
        conversation_rows.append({
            "creator_open_id": creator_open_id,
            "conversation_id": data.get("conversation_id"),
            "creator_im_id": data.get("creator_im_id"),
        })
        print(f"  ✅ {creator_open_id} -> conversation_id: {data.get('conversation_id')}")

    time.sleep(0.5)

print(f"\nDone. {len(conversation_rows)} succeeded, {len(failed_conversations)} failed.")

  ✅ CICFKwAAAACOV4AEeERJl-yievH_HBssqkU7H--75ImCOWkPzQmnsg -> conversation_id: 7659390859917492500
  ✅ QnrT-QAAAACOV4AEeERJl-yievH_HBssaAq-MTUAOiLzfx_rdNpAkg -> conversation_id: 7658276473475924242
  ✅ ctTJ0QAAAACOV4AEeERJl-yievH_HBssp8B68npzmhu7eLphjZP09Q -> conversation_id: 7657528465915248903
  ✅ 5SJoxAAAAACOV4AEeERJl-yievH_HBssnksHd5GVqVNRX-pZpekVRw -> conversation_id: 7658216864620134676
  ✅ JYHREwAAAACOV4AEeERJl-yievH_HBsscEhMRcfgGP6DJPzpaMdC2w -> conversation_id: 7658238340437770516

Done. 5 succeeded, 0 failed.


In [53]:
if conversation_rows:
    df_conversations_new = pd.DataFrame(conversation_rows)
    file_exists = Path(ALL_CONVERSATIONS_NEW_CSV).exists()
    df_conversations_new.to_csv(ALL_CONVERSATIONS_NEW_CSV, mode="a", header=not file_exists, index=False)

    print(f"\nAppended {len(df_conversations_new)} row(s) to {TARGET_COLLAB_CREATORS_CSV}")


Appended 5 row(s) to collabs/target_collaboration_creators.csv


In [54]:
df_conversations_new = pd.read_csv(ALL_CONVERSATIONS_NEW_CSV)

In [55]:
df_all_conversations = pd.concat([df_all_conversations[['creator_im_id', 'conversation_id', 'username']],
           df_conversations_new.merge(df_creators[['username', 'creator_open_id']], how='left', on="creator_open_id")[['creator_im_id', 'conversation_id', 'username']]]).drop_duplicates(subset=['username']).reset_index(drop=True)

In [56]:
df_all_conversations.to_csv(ALL_CONVERSATIONS_CSV, index=False)

# Step 3. Bulk Send Message

In [57]:
# merge all extracted data
df_creators_messaging = df_creators_list \
    .merge(df_creators[['username', 'creator_open_id']], how='left', on="username") \
    .merge(df_target_collab.rename(columns={'name':'target_collaboration_name'}).drop(['batch_name'], axis=1), how='left', on="creator_open_id") \
    .merge(df_all_sample_applications[['creator.creator_open_id', 'status']].rename(columns={'creator.creator_open_id':'creator_open_id', 'status':'sample_status'}), how='left', on="creator_open_id") \
    .merge(df_all_conversations[['creator_im_id', 'conversation_id', 'username']], how='left', on="username")

## Track Message Status

In [58]:
# Update statuses to track
df_creators_messaging['invited_to_join_collab'] = df_creators_messaging['sample_status'].notnull()
df_creators_messaging['invited_to_viber_grp'] = False

In [62]:
df_creators_messaging.loc[df_creators_messaging['sample_status'].notnull(), 'conversation_id']

,Nickname,username,batch_name,creator_open_id,target_collaboration_id,target_collaboration_name,end_time,sample_status,creator_im_id,conversation_id,invited_to_join_collab,invited_to_viber_grp
67,Health Mo,healthmodaily,Health_202606_00k-02k,CICFKwAAAACOV4AEeERJl-yievH_HBssqkU7H--75ImCOW...,NaN,NaN,NaN,COMPLETED,3584883271836707131,7659390859917492500,True,False
160,🪬🧿Delyan finds🧿🪬,labsyan01,Health_202606_00k-02k,XWbLqwAAAACOV4AEeERJl-yievH_HBssqRsLxHvQ2w3Rcj...,7.661986e+18,Health_202606_00k-02k_02,2.101133e+09,CONTENT_PENDING,4542530884813713113,7662617907108724999,True,False
196,tetaytheatay 🇵🇭,tetaytheatay,Health_202606_00k-02k,-sk4ygAAAACOV4AEeERJl-yievH_HBssvuL7MWtK225nPu...,7.661986e+18,Health_202606_00k-02k_02,2.101133e+09,CONTENT_PENDING,6834376724248816298,7663051142813450516,True,False
294,Mommyjane,assortedproductshopping,Health_202606_00k-02k,1-lQUgAAAACOV4AEeERJl-yievH_HBssAPnOFCK4w1ni2r...,7.662397e+18,Health_202606_00k-02k_26,2.101133e+09,CONTENT_PENDING,5273985606040880131,7662707489054408968,True,False
597,Nanie🧿✨,nanie0433,Health_202606_00k-02k,qWs1oAAAAACOV4AEeERJl-yievH_HBssLnbyW_QTioXeoR...,7.662189e+18,Health_202606_00k-02k_15,2.101133e+09,SHIPPED,1425253008333705943,7662759577196314888,True,False
...,...,...,...,...,...,...,...,...,...,...,...,...
13745,arrfindshop,arrfindshop013,All_202606_06k-08k,BYtwmwAAAACOV4AEeERJl-yievH_HBssionha5Lp69ReYB...,7.662539e+18,All_202606_06k-08k_02,2.101133e+09,CONTENT_PENDING,5577645965636830827,7662706748922298632,True,False
13822,NHALYN ᯓᡣ𐭩,nhalyn.11,All_202606_06k-08k,UCwvtAAAAACOV4AEeERJl-yievH_HBssbMO2kXyB_t-bYM...,7.662539e+18,All_202606_06k-08k_02,2.101133e+09,SHIPPED,5148082575162634490,7663080704432308488,True,False
13972,kristineramos17,kristelramos_30,All_202606_06k-08k,DmvsKAAAAACOV4AEeERJl-yievH_HBssxm5k8phBo_8yQz...,7.662539e+18,All_202606_06k-08k_02,2.101133e+09,SHIPPED,4124140782678573640,7663080734069391624,True,False
14347,Don,astra.tala,Health_202606_08k-10k,qqwg9QAAAACOV4AEeERJl-yievH_HBss7DGmliHU_e9KCA...,7.661971e+18,Health_202606_08k-10k_01,2.101133e+09,CONTENT_PENDING,6947531371072728840,7663080809189146901,True,False


In [43]:
df_creators_messaging[~df_creators_messaging['invited_to_join_collab'] & df_creators_messaging['target_collaboration_id'].notnull()]

,Nickname,username,batch_name,creator_open_id,target_collaboration_id,target_collaboration_name,end_time,sample_status,creator_im_id,conversation_id,invited_to_join_collab,invited_to_viber_grp
0,Tito Drew,yourtitodrew,Health_202606_00k-02k,Kx_4sQAAAACOV4AEeERJl-yievH_HBsscdZWAJlmTtwiPj...,7.661986e+18,Health_202606_00k-02k_01,2.101133e+09,NaN,7747447736071296932,7663050355395592466,False,False
1,Doc Alvin,docalvinfrancisco,Health_202606_00k-02k,3H4sbgAAAACOV4AEeERJl-yievH_HBssAZOc-hm3PlOm1i...,7.660854e+18,Health_202606-001,2.101133e+09,NaN,9103083595718770227,7660854457026543892,False,False
2,"Donnamae,RN,RM 1.0",komadronarsdonnamae,Health_202606_00k-02k,QaYdMwAAAACOV4AEeERJl-yievH_HBssXh5A6vjfhZ4IaY...,7.660854e+18,Health_202606-001,2.101133e+09,NaN,1917601662733547789,7660854204488253717,False,False
3,Cris Clerigo💎,mommycris11,Health_202606_00k-02k,xxLCHQAAAACOV4AEeERJl-yievH_HBssjMyu2rYxTzB-U-...,7.660854e+18,Health_202606-001,2.101133e+09,NaN,6196090122478084343,7660854396323397909,False,False
4,Ericka Pineda,erickapineda09,Health_202606_00k-02k,ZQ8XGwAAAACOV4AEeERJl-yievH_HBssNIuQf0uC83GKu1...,7.660854e+18,Health_202606-001,2.101133e+09,NaN,6820943353065492693,7660854565247844629,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...
19336,Michellee,michelleannsormene,Health_202606_10k-12k,VmssLwAAAACOV4AEeERJl-yievH_HBssJiDisTX15mmDa_...,7.661971e+18,Health_202606_10k-12k_01,2.101133e+09,NaN,8595375097484284571,7663080904366358791,False,False
19356,hanieefinds06,blu3berryfinds06,Health_202606_10k-12k,CNeAFwAAAACOV4AEeERJl-yievH_HBssS6LxZUlPTLccRC...,7.661971e+18,Health_202606_10k-12k_01,2.101133e+09,NaN,482788257857509268,7663080930193867016,False,False
19473,marori ❤︎,howeveruwannacallme,Health_202606_10k-12k,TOLwowAAAACOV4AEeERJl-yievH_HBssLm_47qMu9pExC_...,7.661971e+18,Health_202606_10k-12k_01,2.101133e+09,NaN,6851633415167493935,7663080934329532680,False,False
20116,Tita Life 🌸,tipid.tita,All_202606_10k-12k,rvPU_wAAAACOV4AEeERJl-yievH_HBssu77XDSFsZ1iW_V...,7.661970e+18,All_202606_10k-12k_01,2.101133e+09,NaN,619352537878881778,7663080950775677202,False,False


## Send IM Message

In [20]:
message_thank_you = "Thank you so much for accepting our invite, super excited to work with you! Sending over a photo with QR codes para sa content brief and Viber community namin. Just scan para ma access mo!\n\n Feel free to message me here anytime kung may mga tanong ka. Looking forward to creating with you!"

message_viber_qrcode = {
    'url': 'https://p16-oec-sg.ibyteimg.com/tos-alisg-i-aphluv4xwc-sg/de85540122b94ac9bf6c801faf19d01c~tplv-aphluv4xwc-origin-image.image?dr=15570&t=555f072d&ps=933b5bde&shp=5566cfe3&shcp=3c3d9ffb&idc=my&from=1432801251',
    'width': 1280,
    'height': 905
}

In [22]:
send_im_message(conversation_id, "TEXT", {"content": message_thank_you})

{'code': 0,
 'data': {'message_id': '7662761593361745425'},
 'message': 'Success',
 'request_id': '2026071522285969F460EAAD4FF105FE10'}

In [23]:
send_im_message(conversation_id, "IMAGE", message_viber_qrcode)

{'code': 0,
 'data': {'message_id': '7662761637660771848'},
 'message': 'Success',
 'request_id': '202607152229019E28B98FE44D1F061837'}